# Notebook 02 - Repeatability and Reproducibility

**Milestone:** HV-005  
**Technical Gate:** RR-001  
**Primary Engineering Question:** Repeatability and Reproducibility Variance Structure

This notebook evaluates short-term within-test repeatability, matched run-to-run
variation, and matched day-to-day variation using only qualified
`STATIC / CHARACTERIZATION` evidence from the frozen
`VSM-LAB-DATA-v1.1` release.

The analysis is design-aware. The campaign is not assumed to be a classical
fully crossed Part × Operator × Repeat Gage R&R study.


## 1. Engineering Question

How much of the observed static measurement variation is associated with:

1. short-term within-test repeatability;
2. run-to-run differences under matched operating conditions; and
3. day-to-day differences where the experimental design provides genuinely
   matched day coverage?

The analysis must distinguish measurement-system variation from variation
caused by RH, temperature, direction, and experimental design composition.


## 2. Why This Matters

Static characterization established that the DUT has a persistent negative
pre-calibration error together with condition-dependent structure.

However, characterization alone does not identify how much variation is
associated with repeated readings, repeated runs, or repeated test days.

R&R analysis is therefore needed before calibration and uncertainty modeling
so that observed variation is not incorrectly attributed to one source.


## 3. Scope and Analysis Boundary

### Included

- `STATIC / CHARACTERIZATION` observations only;
- repeated samples within Test_ID;
- Test_ID-level run comparisons;
- matched Day 1 / Day 2 comparison where RH coverage is common;
- systematic paired shifts and random paired scatter;
- design limitations and identifiability.

### Deferred

- calibration-model fitting;
- blind validation;
- final Gage R&R acceptance decision;
- dynamic-response repeatability;
- drift/stability analysis;
- final uncertainty budget;
- final DUT compliance decision.

Calibration and validation rows must remain absent from the working frame.


## 4. R&R Terminology Used in This Notebook

**Repeatability** refers to short-term scatter of repeated measurements within
the same Test_ID under nominally stable conditions.

**Run-to-run intermediate precision** refers to differences between Run 1 and
Run 2 when RH, temperature, direction, and day are held matched.

**Day-to-day intermediate precision** is evaluated only where the design
supports a true matched comparison. In this campaign, Days 1 and 2 share RH setpoints
30, 50, and 70 %RH. Day 3 uses different RH setpoints and therefore cannot be
used to estimate an independent three-day effect without confounding RH and
Day.

The roadmap retains the label Repeatability & Reproducibility, but the Run and
Day analyses are interpreted more precisely as **intermediate precision**
because only selected within-system conditions change. Full reproducibility is
not estimated in this notebook.

The analysis remains compatible with Measurement System Analysis / Gage R&R
thinking where supported by the actual design.


## 5. Load Qualified Static Characterization Data

Reusable analyst-side data-loading and summary logic is imported from
`src/static_analysis.py`.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Locate project root and import reusable analyst-side helpers
# ------------------------------------------------------------

current_dir = Path.cwd()
candidate_roots = [current_dir, *current_dir.parents]

project_root = next(
    (
        path
        for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None,
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.static_analysis import (
    add_rh_error_columns,
    load_static_characterization,
    matched_level_pairs,
    paired_difference_metrics,
    pooled_within_test_sd,
    summarize_test_ids,
)

reports_dir = project_root / "reports"
figures_dir = reports_dir / "figures"
tables_dir = reports_dir / "tables"

figures_dir.mkdir(parents=True, exist_ok=True)
tables_dir.mkdir(parents=True, exist_ok=True)

print(f"Project root : {project_root}")
print(f"Figures dir : {figures_dir}")
print(f"Tables dir  : {tables_dir}")


In [ ]:
# ------------------------------------------------------------
# RR-01A: Load qualified STATIC / CHARACTERIZATION data
# ------------------------------------------------------------

characterization_df, test_matrix_df = load_static_characterization(
    project_root
)

characterization_df = add_rh_error_columns(characterization_df)
test_summary_df = summarize_test_ids(characterization_df)

print(f"Characterization rows     : {len(characterization_df):,}")
print(f"Characterization Test_IDs : {characterization_df['Test_ID'].nunique():,}")
print(f"Test_ID summary rows      : {len(test_summary_df):,}")

display(test_summary_df.head())


In [ ]:
# ------------------------------------------------------------
# RR-01B: Verify analysis boundary
# ------------------------------------------------------------

calibration_rows_in_working_frame = (
    characterization_df["Analysis_Role"].eq("CALIBRATION").sum()
)

validation_rows_in_working_frame = (
    characterization_df["Analysis_Role"].eq("VALIDATION").sum()
)

rr01_boundary_pass = (
    len(characterization_df) > 0
    and characterization_df["Test_Family"].eq("STATIC").all()
    and characterization_df["Analysis_Role"].eq("CHARACTERIZATION").all()
    and calibration_rows_in_working_frame == 0
    and validation_rows_in_working_frame == 0
)

print(f"STATIC only               : {characterization_df['Test_Family'].eq('STATIC').all()}")
print(f"CHARACTERIZATION only     : {characterization_df['Analysis_Role'].eq('CHARACTERIZATION').all()}")
print(f"CALIBRATION rows          : {calibration_rows_in_working_frame}")
print(f"VALIDATION rows           : {validation_rows_in_working_frame}")
print(f"RR-01 boundary check      : {'PASS' if rr01_boundary_pass else 'FAIL'}")

if not rr01_boundary_pass:
    raise RuntimeError("RR-01 analysis-boundary verification failed.")


## 6. Experimental-Design Audit

Before estimating R&R quantities, the replication structure must be verified.

A marginal comparison of Day 1, Day 2, and Day 3 would be misleading because
Day is not fully crossed with RH.


In [ ]:
# ------------------------------------------------------------
# RR-01C: Verify samples per Test_ID and factor coverage
# ------------------------------------------------------------

sample_counts = (
    characterization_df.groupby("Test_ID")
    .size()
    .rename("N_Samples")
)

factor_coverage = {
    "RH_Setpoint": sorted(test_summary_df["RH_Setpoint"].unique()),
    "Temperature_Setpoint_C": sorted(
        test_summary_df["Temperature_Setpoint_C"].unique()
    ),
    "Direction": sorted(test_summary_df["Direction"].unique()),
    "Day_ID": sorted(test_summary_df["Day_ID"].unique()),
    "Run_ID": sorted(test_summary_df["Run_ID"].unique()),
}

print("Samples per Test_ID:")
display(sample_counts.value_counts().sort_index().rename("N_Test_IDs"))

print("\nFactor coverage:")
for factor, values in factor_coverage.items():
    print(f"{factor}: {values}")

rr01_sample_structure_pass = (
    len(test_summary_df) == 120
    and sample_counts.eq(60).all()
)

print(
    "\n120 Test_IDs / 60 samples per Test_ID : "
    f"{'PASS' if rr01_sample_structure_pass else 'FAIL'}"
)

day_by_rh_df.to_csv(
    tables_dir / "hv005_day_by_rh_design_audit.csv",
    index=False,
)


In [ ]:
# ------------------------------------------------------------
# RR-01D: Audit Run 1 / Run 2 matching structure
# ------------------------------------------------------------

run_match_keys = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
    "Day_ID",
]

run_structure_df = (
    test_summary_df
    .groupby(run_match_keys)
    .agg(
        N_Test_IDs=("Test_ID", "nunique"),
        N_Runs=("Run_ID", "nunique"),
    )
    .reset_index()
    .sort_values(run_match_keys)
)

complete_run_conditions = (
    run_structure_df["N_Test_IDs"].eq(2)
    & run_structure_df["N_Runs"].eq(2)
)

print(f"Run-match conditions      : {len(run_structure_df)}")
print(f"Complete Run 1/2 pairs    : {complete_run_conditions.sum()}")

rr01_run_structure_pass = (
    len(run_structure_df) == 60
    and complete_run_conditions.all()
)

print(
    "Run structure check       : "
    f"{'PASS' if rr01_run_structure_pass else 'FAIL'}"
)


In [ ]:
# ------------------------------------------------------------
# RR-01E: Audit Day × RH crossing structure
# ------------------------------------------------------------

day_by_rh_df = (
    test_summary_df
    .groupby("RH_Setpoint")
    .agg(
        N_Days=("Day_ID", "nunique"),
        Days=("Day_ID", lambda x: ", ".join(map(str, sorted(x.unique())))),
        N_Test_IDs=("Test_ID", "nunique"),
    )
    .reset_index()
    .sort_values("RH_Setpoint")
)

display(day_by_rh_df)

day12_common_rh = sorted(
    set(
        test_summary_df.loc[
            test_summary_df["Day_ID"] == 1,
            "RH_Setpoint",
        ]
    )
    & set(
        test_summary_df.loc[
            test_summary_df["Day_ID"] == 2,
            "RH_Setpoint",
        ]
    )
)

day3_rh = sorted(
    test_summary_df.loc[
        test_summary_df["Day_ID"] == 3,
        "RH_Setpoint",
    ].unique()
)

print(f"Common RH for Day 1/2 : {day12_common_rh}")
print(f"RH represented on Day 3: {day3_rh}")

rr01_partial_crossing_documented = (
    day12_common_rh == [30.0, 50.0, 70.0]
    and day3_rh == [20.0, 40.0, 60.0, 80.0]
)

print(
    "Partial Day × RH crossing verified : "
    f"{rr01_partial_crossing_documented}"
)


### RR-01 Design Interpretation

The design audit must be completed before any day-effect estimate is accepted.

Expected structure:

- 120 Test_IDs;
- 60 samples per Test_ID;
- 60 complete Run 1 / Run 2 matched conditions;
- Day 1 / Day 2 common RH setpoints: 30, 50, 70 %RH;
- Day 3 RH setpoints: 20, 40, 60, 80 %RH.

This partial crossing means that Day 3 cannot be interpreted as an independent
day effect relative to Days 1 and 2.


## 7. RR-02 — Within-Test Repeatability

Within-test repeatability is quantified from the short-term scatter of
`Error_RH` within each Test_ID.

The pooled within-test standard deviation is:

$$
s_{repeat}
=
\sqrt{
\frac{
\sum_i (n_i-1)s_i^2
}{
\sum_i (n_i-1)
}
}
$$

where (s_i) is the within-Test_ID standard deviation and (n_i) is the
number of repeated samples in Test_ID (i).


In [ ]:
# ------------------------------------------------------------
# RR-02A: Test_ID-level within-test repeatability statistics
# ------------------------------------------------------------

repeatability_summary = (
    test_summary_df["Std_Error_RH"]
    .describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
)

pooled_repeatability_sd = pooled_within_test_sd(test_summary_df)

print("Within-Test_ID SD distribution (%RH):")
display(repeatability_summary.to_frame("Std_Error_RH").round(6))

print(
    f"\nPooled within-test repeatability SD : "
    f"{pooled_repeatability_sd:.6f} %RH"
)

rr02_repeatability_pass = (
    np.isfinite(pooled_repeatability_sd)
    and pooled_repeatability_sd >= 0
)

print(
    "RR-02 repeatability calculation      : "
    f"{'PASS' if rr02_repeatability_pass else 'FAIL'}"
)

test_summary_df[
    [
        "Test_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Day_ID",
        "Run_ID",
        "N_Samples",
        "Mean_Error_RH",
        "Std_Error_RH",
        "Range_Error_RH",
    ]
].to_csv(
    tables_dir / "hv005_test_id_repeatability_summary.csv",
    index=False,
)


In [ ]:
# ------------------------------------------------------------
# RR-02B: Inspect repeatability across operating conditions
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(8, 5))

for temperature in sorted(
    test_summary_df["Temperature_Setpoint_C"].unique()
):
    subset = test_summary_df.loc[
        test_summary_df["Temperature_Setpoint_C"] == temperature
    ]

    ax.scatter(
        subset["RH_Setpoint"],
        subset["Std_Error_RH"],
        alpha=0.7,
        label=f"{temperature:g} °C",
    )

ax.set_xlabel("RH Setpoint (%RH)")
ax.set_ylabel("Within-Test_ID Error SD (%RH)")
ax.set_title("Short-Term Repeatability across Static Conditions")
ax.grid(True, alpha=0.3)
ax.legend(title="Temperature")

fig.savefig(
    figures_dir / "hv005_repeatability_by_rh_temperature.png",
    dpi=200,
    bbox_inches="tight",
)

plt.show()


### Serial-Correlation Check

The 60 samples within each Test_ID are sequential time samples, not guaranteed
independent draws.

Therefore this notebook checks lag-1 autocorrelation before interpreting
(s_{repeat}/\sqrt{60}) as the standard deviation of a 60-sample mean.


In [ ]:
# ------------------------------------------------------------
# RR-02C: Lag-1 autocorrelation within each Test_ID
# ------------------------------------------------------------

lag1_autocorr_df = (
    characterization_df
    .sort_values(["Test_ID", "Elapsed_s"])
    .groupby("Test_ID")["Error_RH"]
    .apply(lambda x: x.autocorr(lag=1))
    .rename("Lag1_Autocorrelation")
    .reset_index()
)

display(
    lag1_autocorr_df["Lag1_Autocorrelation"]
    .describe(
        percentiles=[0.05, 0.25, 0.50, 0.75, 0.95]
    )
    .to_frame()
    .round(4)
)

print(
    "Naive SD(mean) under independence assumption : "
    f"{pooled_repeatability_sd / np.sqrt(60):.6f} %RH"
)

print(
    "\nInterpret the value above only after reviewing serial correlation."
)


### RR-02 Engineering Interpretation

Complete after executing RR-02.

Questions:

- What is the pooled within-test SD?
- Is repeatability approximately stable across RH and temperature?
- Is serial correlation negligible, moderate, or substantial?
- Is the independence-based (s/\sqrt{n}) reduction defensible?


## 8. RR-03 — Matched Run-to-Run Intermediate Precision

Run 1 and Run 2 are compared only after matching:

- RH setpoint;
- temperature setpoint;
- direction; and
- day.

The paired difference is:

$$
\Delta e_{run}
=
\bar e_{Run2}
-
\bar e_{Run1}
$$

A non-zero mean paired difference is a systematic run shift and must be
reported separately from the scatter of the paired differences.


In [ ]:
# ------------------------------------------------------------
# RR-03A: Create complete Run 1 / Run 2 matched pairs
# ------------------------------------------------------------

run_pairs_df = matched_level_pairs(
    test_summary_df,
    match_keys=run_match_keys,
    level_column="Run_ID",
    level_a=1,
    level_b=2,
    value_column="Mean_Error_RH",
    label_a="Mean_Error_Run1",
    label_b="Mean_Error_Run2",
)

display(run_pairs_df.head(10).round(4))

print(run_pairs_df["Pair_Complete"].value_counts(dropna=False))

rr03_run_pairs_pass = (
    len(run_pairs_df) == 60
    and run_pairs_df["Pair_Complete"].all()
)

print(
    "\nRR-03 matched run pairs : "
    f"{'PASS' if rr03_run_pairs_pass else 'FAIL'}"
)


In [ ]:
# ------------------------------------------------------------
# RR-03B: Run paired-difference metrics
# ------------------------------------------------------------

run_metrics = paired_difference_metrics(run_pairs_df)
run_metrics_df = pd.DataFrame([run_metrics])

display(run_metrics_df.round(6))

run_metrics_df.to_csv(
    tables_dir / "hv005_run_pair_metrics.csv",
    index=False,
)


In [ ]:
# ------------------------------------------------------------
# RR-03C: Run difference by RH
# ------------------------------------------------------------

run_by_rh_df = (
    run_pairs_df.loc[run_pairs_df["Pair_Complete"]]
    .groupby("RH_Setpoint")
    .agg(
        N_Pairs=("Difference", "size"),
        Mean_Run2_minus_Run1=("Difference", "mean"),
        Median_Run2_minus_Run1=("Difference", "median"),
        Std_Run2_minus_Run1=("Difference", "std"),
    )
    .reset_index()
    .sort_values("RH_Setpoint")
)

display(run_by_rh_df.round(6))

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    run_pairs_df["RH_Setpoint"],
    run_pairs_df["Difference"],
    alpha=0.7,
    label="Matched Run 2 - Run 1",
)

ax.plot(
    run_by_rh_df["RH_Setpoint"],
    run_by_rh_df["Mean_Run2_minus_Run1"],
    marker="o",
    linewidth=2,
    label="Mean paired difference",
)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("RH Setpoint (%RH)")
ax.set_ylabel("Run 2 - Run 1 Mean Error (%RH)")
ax.set_title("Matched Run-to-Run Error Difference")
ax.grid(True, alpha=0.3)
ax.legend()

fig.savefig(
    figures_dir / "hv005_matched_run_difference.png",
    dpi=200,
    bbox_inches="tight",
)

run_by_rh_df.to_csv(
    tables_dir / "hv005_run_difference_by_rh.csv",
    index=False,
)

plt.show()


### RR-03 Engineering Interpretation

Complete after execution.

Separate:

1. **systematic run shift** — the mean of the paired Run 2 − Run 1
   differences; and
2. **run-to-run scatter** — the standard deviation of the paired differences.

Do not collapse these into one quantity without stating the measurement model.


## 9. RR-04 — Matched Day-to-Day Intermediate Precision

Day is only compared where the same RH conditions exist on both days.

The analysis therefore uses only:

- Day 1;
- Day 2;
- RH = 30, 50, 70 %RH.

Run 1 and Run 2 are first averaged within each RH × Temperature × Direction ×
Day condition. This prevents the day comparison from being driven by a
particular run label.


In [ ]:
# ------------------------------------------------------------
# RR-04A: Build day-condition means from the matched Day 1 / Day 2 subset
# ------------------------------------------------------------

day12_test_df = test_summary_df.loc[
    test_summary_df["Day_ID"].isin([1, 2])
    & test_summary_df["RH_Setpoint"].isin([30.0, 50.0, 70.0])
].copy()

day_condition_df = (
    day12_test_df
    .groupby(
        [
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Direction",
            "Day_ID",
        ],
        as_index=False,
    )
    .agg(
        Mean_Error_RH=("Mean_Error_RH", "mean"),
        N_Runs=("Run_ID", "nunique"),
    )
)

display(day_condition_df.head(12).round(4))

all_day_conditions_have_two_runs = (
    day_condition_df["N_Runs"].eq(2).all()
)

print(
    "Every Day-condition contains Run 1 and Run 2 : "
    f"{all_day_conditions_have_two_runs}"
)


In [ ]:
# ------------------------------------------------------------
# RR-04B: Create matched Day 1 / Day 2 pairs
# ------------------------------------------------------------

day_match_keys = [
    "RH_Setpoint",
    "Temperature_Setpoint_C",
    "Direction",
]

day_pairs_df = matched_level_pairs(
    day_condition_df,
    match_keys=day_match_keys,
    level_column="Day_ID",
    level_a=1,
    level_b=2,
    value_column="Mean_Error_RH",
    label_a="Mean_Error_Day1",
    label_b="Mean_Error_Day2",
)

display(day_pairs_df.head(10).round(4))

rr04_day_pairs_pass = (
    len(day_pairs_df) == 18
    and day_pairs_df["Pair_Complete"].all()
    and all_day_conditions_have_two_runs
)

print(
    "\nRR-04 matched Day 1/2 pairs : "
    f"{'PASS' if rr04_day_pairs_pass else 'FAIL'}"
)


In [ ]:
# ------------------------------------------------------------
# RR-04C: Day paired-difference metrics
# ------------------------------------------------------------

day_metrics = paired_difference_metrics(day_pairs_df)
day_metrics_df = pd.DataFrame([day_metrics])

display(day_metrics_df.round(6))

day_metrics_df.to_csv(
    tables_dir / "hv005_day_pair_metrics.csv",
    index=False,
)


In [ ]:
# ------------------------------------------------------------
# RR-04D: Day difference by RH
# ------------------------------------------------------------

day_by_rh_metrics_df = (
    day_pairs_df.loc[day_pairs_df["Pair_Complete"]]
    .groupby("RH_Setpoint")
    .agg(
        N_Pairs=("Difference", "size"),
        Mean_Day2_minus_Day1=("Difference", "mean"),
        Median_Day2_minus_Day1=("Difference", "median"),
        Std_Day2_minus_Day1=("Difference", "std"),
    )
    .reset_index()
    .sort_values("RH_Setpoint")
)

display(day_by_rh_metrics_df.round(6))

fig, ax = plt.subplots(figsize=(8, 5))

ax.scatter(
    day_pairs_df["RH_Setpoint"],
    day_pairs_df["Difference"],
    alpha=0.75,
    label="Matched Day 2 - Day 1",
)

ax.plot(
    day_by_rh_metrics_df["RH_Setpoint"],
    day_by_rh_metrics_df["Mean_Day2_minus_Day1"],
    marker="o",
    linewidth=2,
    label="Mean paired difference",
)

ax.axhline(0, linestyle="--", linewidth=1)
ax.set_xlabel("RH Setpoint (%RH)")
ax.set_ylabel("Day 2 - Day 1 Mean Error (%RH)")
ax.set_title("Matched Day-to-Day Error Difference")
ax.grid(True, alpha=0.3)
ax.legend()

fig.savefig(
    figures_dir / "hv005_matched_day_difference.png",
    dpi=200,
    bbox_inches="tight",
)

day_by_rh_metrics_df.to_csv(
    tables_dir / "hv005_day_difference_by_rh.csv",
    index=False,
)

plt.show()


### RR-04 Engineering Interpretation

Complete after execution.

Important limitation:

Day 3 is not included in the independent day-effect estimate because its RH
coverage does not overlap with Days 1 and 2.

This is a design limitation, not missing-data cleaning.


## 10. RR-05 — R&R Synthesis

This section will combine the three defensible views of measurement variation:

- pooled short-term within-test repeatability;
- matched run-to-run shift and scatter;
- matched Day 1 / Day 2 shift and scatter.

The synthesis must keep systematic shifts separate from random scatter.


In [ ]:
# ------------------------------------------------------------
# RR-05A: Consolidated R&R summary
# ------------------------------------------------------------

rr_summary_df = pd.DataFrame(
    [
        {
            "Component": "Within-Test Repeatability",
            "Statistic": "Pooled SD",
            "Value_RH": pooled_repeatability_sd,
            "N": len(test_summary_df),
        },
        {
            "Component": "Run-to-Run",
            "Statistic": "Mean Run2-Run1 shift",
            "Value_RH": run_metrics["Mean_Difference"],
            "N": run_metrics["N_Matched_Pairs"],
        },
        {
            "Component": "Run-to-Run",
            "Statistic": "SD of paired differences",
            "Value_RH": run_metrics["Std_Difference"],
            "N": run_metrics["N_Matched_Pairs"],
        },
        {
            "Component": "Day-to-Day",
            "Statistic": "Mean Day2-Day1 shift",
            "Value_RH": day_metrics["Mean_Difference"],
            "N": day_metrics["N_Matched_Pairs"],
        },
        {
            "Component": "Day-to-Day",
            "Statistic": "SD of paired differences",
            "Value_RH": day_metrics["Std_Difference"],
            "N": day_metrics["N_Matched_Pairs"],
        },
    ]
)

display(rr_summary_df.round(6))

rr_summary_df.to_csv(
    tables_dir / "hv005_rr_summary.csv",
    index=False,
)


### Exported Evidence

Key R&R figures and summary tables are written to:

- `reports/figures/`
- `reports/tables/`

These exported artifacts are intended for technical review, the final report,
and the Analytics Day poster. Notebook outputs remain the analytical record,
while the exported files provide stable presentation evidence.


## 11. Design Limitations

The static campaign supports strong matched Run 1 / Run 2 analysis but only a
limited matched Day 1 / Day 2 analysis.

Therefore:

- run-to-run structure can be evaluated across all available static conditions;
- day-to-day structure can be evaluated only for the shared RH subset;
- Day 3 cannot be separated from RH composition;
- only two day levels are available in the matched day subset;
- only two run levels are available per condition;
- repeated 1 Hz samples may contain serial correlation.

These limitations constrain how strongly variance components can be
generalized.


## 12. Engineering Interpretation

Complete after numerical execution.

The interpretation should answer:

1. What is the magnitude of short-term repeatability?
2. Is there a systematic Run 2 − Run 1 shift?
3. How variable are the run differences across operating conditions?
4. Is there a systematic Day 2 − Day 1 shift on the matched subset?
5. Is the day result stable across RH = 30, 50, 70 %RH?
6. Which effects should later be corrected, controlled, or carried into the
   uncertainty model?
7. What cannot be concluded because of the partial Day × RH crossing?


## 13. Connection to Calibration and Uncertainty

HV-005 does not fit the calibration model.

Its purpose is to identify measurement-variation structure that later affects:

- calibration-model interpretation;
- residual-error analysis;
- repeatability uncertainty;
- reproducibility uncertainty;
- measurement procedure controls; and
- final compliance decision rules.

Observed systematic run/day shifts are not automatically converted into
standard uncertainty components. Their treatment depends on whether the
effect is controlled, corrected, modeled, or left as residual variability.


## 14. RR-001 Gate

RR-001 will be closed only after:

- all design checks pass;
- repeatability is quantified;
- run matching is complete;
- the matched Day 1 / Day 2 subset is verified;
- calibration and validation leakage remains zero;
- no hidden simulator truth is used;
- systematic shifts and random scatter are interpreted separately; and
- the complete notebook reproduces from a clean kernel.

Possible outcomes:

- **PASS**
- **PASS WITH DOCUMENTED CONDITIONS**
- **FAIL**
